### FTP API data grab

This file contains the code for downloading a FTP transcription export from their API

Still a work in progress

I am using Python's requests library because I have more facility in Python and Windows PowerShell is a curse that I prefer not to suffer under when I can avoid it. If on a MacOS or Linux/Unix system, the documentation on the FTP website explains how to accomplish this from the command line: https://content.fromthepage.com/project-owner-documentation/api-keys/ 

In [1]:
# hiding the API key
import os
import dotenv

# change to the directory where the dotenv file is (unique for each person)
os.chdir("/Users/charl/JBPP")

# load in stuff hidden in the .env file
dotenv.load_dotenv()
JBPP_key = os.getenv('JBPP_key')

In [2]:
# import required packages
import requests
import pandas as pd
import json
import re

# code to create post request
apikey = JBPP_key

root = "http://fromthepage.com/iiif"
endpoint = "/collection/ready-to-transfer-to-drupal" # this endpoint is the only thing that needs editing 
# use IIIF slug found at bottom of "export" tab in FTP document set you want to export from
headers = {"Authorization": f"Token token={apikey}"}

In [23]:
# submit post request using requests library (operates same as cURL, just in Python)
response = requests.post(root+endpoint, headers=headers)


In [24]:
# to run if you wanna look at the raw text or check status

print(response.status_code)
# should be 200
print(response.text)

200
{
  "@context": "http://iiif.io/api/presentation/2/context.json",
  "@id": "https://fromthepage.com/iiif/set/ready-to-transfer-to-drupal",
  "@type": "sc:Collection",
  "label": "Ready to Transfer to Drupal",
  "manifests": [
    {
      "@id": "https://fromthepage.com/iiif/32076530/manifest",
      "@type": "sc:Manifest",
      "label": "Draft From Julian Bond to George Booker, ca. 7 July 1967",
      "metadata": [
        {
          "label": "dc:source",
          "value": "1207"
        }
      ],
      "service": {
        "@context": "http://www.fromthepage.org/jsonld/1/context.json",
        "@id": "https://fromthepage.com/iiif/32076530/status",
        "label": "Work Status",
        "profile": "https://github.com/benwbrum/fromthepage/wiki/FromThePage-Support-for-the-IIIF-Presentation-API-and-Web-Annotations#service",
        "pctComplete": 100.0,
        "pctTranscribed": 100.0,
        "pctOcrCorrected": 0.0,
        "pctIndexed": 0,
        "pctMarkedBlank": 0,
        "

In [25]:
# convert to dataframe using json_normalize
# record_path=['manifests'] is to ignore metadata associated with the API call that's returned in the response
# but is not connected to the actual doc set content
response_df = pd.json_normalize(json.loads(response.text), record_path=['manifests'])

In [11]:
# if interested in taking a look at the dataframe:

# response_df.head()
# I'm curious if anything else will become metadata because there's more metadata in the bulk uploaded documents
# If it breaks later, refer to that column to see how to fix it

In [7]:
# don't run this cell (you can, you just don't need to)

#bunch_of_tuples = [] # this is where we'll store all the key (PJB ID) - value (Document Body) pairs to convert to a dataframe

#for i in range(len(response_df)): # iterates over each row of the dataframe - 
#    # there are other ways to do this but it's not prohibitively inefficient
#    url = response_df['@id'][i] # indexes into the value in the first column of the dataframe (the IIIF url)
#    cut = url.split('/')[4] # slices out the unique work_id - to be used to locate plaintext export
#    try:
#        pjb_id = response_df['metadata'][i][0]['value'] # tries to make key based on identifier aka PJB ID
#    except TypeError:
#       pjb_id = cut # if it fails, it instead makes key on the basis of the work_id (guaranteed to be unique)
#    new_url = f'https://fromthepage.com/iiif/{cut}/export/plaintext/verbatim' # url that hosts the plaintext export
#    final = requests.get(new_url) # get request on plaintext export url
#    bunch_of_tuples.append((pjb_id, final.text)) # appends key-value pair to dictionary
    

In [26]:
# testing for XHTML compatibility (I think it might be better than plaintext)
bunch_of_tuples = [] # this is where we'll store all the variable pairs (PJB ID, Document Body) to convert to a dataframe

for i in range(len(response_df)): # iterates over each row of the dataframe - 
    # there are other ways to do this but it's not prohibitively inefficient
    url = response_df['@id'][i] # indexes into the value in the first column of the dataframe (the IIIF url)
    cut = url.split('/')[4] # slices out the unique work_id - to be used to locate html export
    try:
        pjb_id = response_df['metadata'][i][0]['value'] # tries to make key based on idenitifier aka PJB ID
    except TypeError:
        pjb_id = cut # if it fails, it instead makes key on the basis of the work_id (guaranteed to be unique)
    new_url = f'https://fromthepage.com/iiif/{cut}/export/html' # url that hosts the html export
    final = requests.get(new_url) # get request on html export url
    html = final.text
    title_position = html.find('<title>')
    desired_content = html[title_position:] 
    bunch_of_tuples.append((pjb_id, desired_content))# appends to list of tuples

In [125]:
# if you wanna check out the dictionary (should be same length as response_df)

# bunch_of_tuples

In [27]:
# create dataframe, label columns + set PJB ID as index
df_final = pd.DataFrame(bunch_of_tuples, columns=['PJB ID', 'Document Body'])

In [28]:
df_final

,PJB ID,Document Body
0,1207,<title> Draft From Julian Bond to George Booke...
1,1253,<title> Draft From Julian Bond to S. W. Graydo...
2,970,"<title> From Julian Bond to Barry Hoffman, 20 ..."
3,1075,"<title> From Julian Bond to Esaias Lee, 13 Jun..."
4,1246,"<title> From Julian Bond to Shepherd Burge, Jr..."
5,921,"<title> To Julian Bond from Fred Vinson, Jr., ..."
6,1056,"<title> To Julian Bond from Ken Kirkpatrick, 7..."
7,PJB 1497,"<title> To Julian Bond from Marvin Wall, 5 Jan..."
8,PJB777,"<title> To Julian Bond from Richard Jackson, 1..."


In [10]:
df_final[df_final['PJB ID'].isna()]

,PJB ID,Document Body


In [19]:
# to check how it looks with a two page doc
# newlist = [x for x in list if "To Julian Bond from Margaret Linton" in x]
# newlist

In [29]:
# if you wanna take a look at the final dataframe

mylist = df_final['Document Body'].tolist()
mylist[0]
# I wonder if the title tags would break this

'<title> Draft From Julian Bond to George Booker, ca. 7 July 1967</title>\n    </head>\n\n    <body>\n    <h1 class="work-title">Draft From Julian Bond to George Booker, ca. 7 July 1967</h1>\n    <div class="export-metadata">FromThePage export of Draft From Julian Bond to George Booker, ca. 7 July 1967 from The Papers of Julian Bond made on 2025-09-30 19:21:20 +0000.\n        <p>Identifier: 1207</p>\n      <p>\n        FromThePage version: 22.10\n      </p>\n    </div>\n\n    <hr />\n    <h2 class="divider">Page Transcripts</h2>\n\n    <div class="pages">\n      <div id="page-33553064">\n        <h3><a name="page-33553064">1</a></h3>\n        <div class="page-content">\n              <p>George Booker <br/>\nTwin Towers <br/>\n2020 North Atlantic <br/>\nApartment 201 South <br/>\nCocoa Beach <br/>\nFlorida  32931</p>\n\n<p>Dear George:</p>\n\n<p>Thanks for your letter. I look forward to getting the coat and then completing the wardrobe.</p>\n\n<p>I think I\'m going to be in Miami during

NEW: "username" now reflects preferred attribution name for all contributors instead of just username. Yay!

In [30]:
editors_tag = 'small'
editors_tag_class = ' class="page-version-username"'
# title_tag = 'title'
# title_tag_class = ''
# I think I would prefer to match on title even if PJB ID is more robust because title is easier to extrac
content_tag = 'div'
content_tag_class = ' class="page-content"'
tags = {editors_tag: editors_tag_class,
       # title_tag: title_tag_class,
       content_tag: content_tag_class}
final_list = []

for i in range(len(mylist)):
    dirty = mylist[i]
    dictionary = {}
    for tag, tag_class in tags.items():
        reg_str = "<" + tag + tag_class + ">(.*?)</" + tag + ">"
        res = re.findall(reg_str, dirty, re.DOTALL)
        key = tag
        dictionary[key] = res
    dictionary['small'] = set(dictionary['small'])
    for k,v in dictionary.items():
        target = ' '.join(v)
        target_stage_2 = target.replace('\n','')
        dictionary[k] = target_stage_2.strip(' ')
    check = dictionary['div'] + "<p>Thanks to FromThePage transcription contributors: " + dictionary['small'] + "</p>"
    final_list.append(check)
    

In [31]:
df_final['Document Body'] = final_list

In [32]:
id_list = df_final['PJB ID'].to_list()
new_ids = []
for id in id_list:
    # Remove any spaces or PJBs for standardization
    id = id.replace(" ", "")
    id = id.replace("PJB", "")
    id = id.replace("PB", "")
    new_ids.append(id)
        
print(new_ids)
df_final['PJB ID'] = new_ids
df_final

['1207', '1253', '970', '1075', '1246', '921', '1056', '1497', '777']


,PJB ID,Document Body
0,1207,<p>George Booker <br/>Twin Towers <br/>2020 No...
1,1253,<p>Mr. S. W. Graydon <br/><strike>Department o...
2,970,"<p>March 20, 1967</p><p>Mr. Barry Hoffman<br/>..."
3,1075,"<p>June 13, 1967</p><p>Rev. Esaias F. Lee <br/..."
4,1246,"<p>July 24, 1967</p><p>Mrs. Shephard Burge <br..."
5,921,"<p>March 7, 1967</p><p>Honorable Julian Bond<b..."
6,1056,<p>American Friends Service Committee<br/>Paci...
7,1497,"<p>January 5, 1968</p><p>The Honorable Julian ..."
8,777,"<p>P. O. Box 321<br/>Union Springs, Alabama<br..."


In [39]:
example = df_final[df_final['PJB ID'] == 'PJB 2068']
print(example['Document Body'][134])

<p>WESTERN UNION<br/>TELEGRAM</p><p>1104P EDT AUG 30 68 AC504 K616</p><p>WZA467 NL PD WG WELLINGTON KANS 30<br/>HONORABLE JULIAN BOND<br/>162 EAST EUHARLEE ST SOUTHWEST ATLA<br/>CONGRATULATIONS ON MAGNIFICIENT PERFORMANCE IN CHICAGO.  KANSAS<br/>STATE CONFERENCE OF NAACP BRANCHES CORDIALLY INVIT YOU TO ADDRESS<br/>ITS ANNUAL FREEDOM FUN DINNER TO BE HELD IN TOPEKA KANSAS<br/>NOV 16 1968 AT 7PM.  ALTHOUGH YOU HAVE MADE PUBLIC APPEARANCES<br/>IN MANY SECTIONS OF COUNTRY NO ONE IN KANSAS CAN REMEMBER YOUR<br/>EVER VISITING STATE.  FOR THIS AND OTHR REASONS WE URGE YOU<br/>TO SERIOUSLY CONSIDER ACCEPTING OUR INVIGATIONS.<br/>YOU MAY CALL OR WIRE COLLECT<br/>DR CHARLES ROQUEMORE PRESIDENT<br/>220 EAST KANSAS AVE<br/>WELLINGTON KANS FA 64691</p>                       <p>Dear ........:<br/>Please forgive me for taking so long to answer your kind telegram.<br/>I am sorry but I cannot come to Kansas on the date you have indicated.<br/>My schedule is a busy one, and I am trying to limit my time 

This presents a particularly interesting problem, but one that I imagine solving would simply cause more problems than solutions. In using this new method, line breaks (`<br/>`) hold together instead of disappearing. This is good when we want to use linebreaks (frequently in Series II, for addresses) but bad when contributors literally represent line breaks (against transcription guidelines). But I suppose it can always be sorted out in proofreading.

In [33]:
# export to csv for transfer to Drupal!
import datetime
# just gonna do UTC minus 5 because timezones are a pain, and this doesn't need to be perfect
# so sometimes it'll be CDT and sometimes EST but so be it
date = datetime.datetime.now(datetime.UTC) - datetime.timedelta(hours = 5) 
print(f'{date:%Y%m%d}')
df_final.to_csv(f'export_to_drupal_{date:%m%d%Y}.csv', index=False)

20250930


In [17]:
os.getcwd()

'C:\\Users\\charl\\JBPP'

In [18]:
mod_states = pd.read_csv('content_moderation_state_field_data.csv')

In [19]:
mod_states

,id,revision_id,langcode,uid,workflow,moderation_state,content_entity_type_id,content_entity_id,content_entity_revision_id,default_langcode,revision_translation_affected
0,6289,86575,en,1,document_workflow,proof_read,node,677,86621,1,1
1,6288,86574,en,1,document_workflow,proof_read,node,678,86620,1,1
2,6287,86573,en,1,document_workflow,proof_read,node,679,86619,1,1
3,6286,86572,en,1,document_workflow,proof_read,node,680,86618,1,1
4,6285,86571,en,1,document_workflow,proof_read,node,681,86617,1,1
...,...,...,...,...,...,...,...,...,...,...,...
3468,2822,93525,en,1,document_workflow,draft,node,4149,93648,1,1
3469,2821,93526,en,1,document_workflow,draft,node,4150,93649,1,1
3470,2820,93527,en,1,document_workflow,draft,node,4151,93650,1,1
3471,2819,89373,en,1,document_workflow,draft,node,4152,89433,1,1


In [21]:
for idx, row in mod_states.iterrows():
    if (row['moderation_state'] == 'cataloged') or (row['moderation_state'] == 'digitized'):
        print(row['content_entity_id'], row['workflow'])

726 document_workflow
777 document_workflow
921 document_workflow
970 document_workflow
1056 document_workflow
1075 document_workflow
1207 document_workflow
1246 document_workflow
1253 document_workflow
1497 document_workflow
2309 document_workflow
2310 document_workflow
2311 document_workflow
2312 document_workflow
2313 document_workflow
2314 document_workflow
2315 document_workflow
2316 document_workflow


In [22]:
mod_states.moderation_state.value_counts()

moderation_state
draft           2921
proof_read       526
cataloged         17
early_access       7
digitized          1
tandem_read        1
Name: count, dtype: int64